# Sign Language Detection — YOLOv5 Training Pipeline

This Jupyter Notebook provides the complete end-to-end workflow to train a **YOLOv5** object detection model on the **American Sign Language (ASL) Alphabet dataset** using Google Colab's free GPU acceleration.

### Pipeline Steps:
1. **GPU Verification**
2. **YOLOv5 Setup & Dependencies**
3. **Dataset Acquisition (Roboflow / YOLO format)**
4. **Model Training (`yolov5s.pt`, 416x416, 50 epochs)**
5. **Model Evaluation (mAP@0.5, Confusion Matrix, Validation)**
6. **Inference Verification**
7. **Export `best.pt` Weights for Web Deployment**

## 1. Verify GPU Availability
Ensure runtime type is set to GPU in Google Colab: `Runtime` -> `Change runtime type` -> `T4 GPU`.

In [ ]:
!nvidia-smi

## 2. Clone YOLOv5 & Install Dependencies

In [ ]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
%pip install -qr requirements.txt

import torch
from IPython.display import Image, clear_output
print(f'Setup complete. Using torch {torch.__version__} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')

## 3. Download the ASL Alphabet Dataset
We use the ASL Fingerspelling Alphabet dataset in YOLOv5 PyTorch format from Roboflow Universe.
Replace `YOUR_API_KEY` below with your Roboflow API Key, or upload your `data.yaml` and dataset images directly.

In [ ]:
!pip install -q roboflow

from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
# project = rf.workspace("david-lee").project("american-sign-language-letters")
# dataset = project.version(1).download("yolov5")

## 4. Train YOLOv5 on Sign Language Dataset
We train on `416x416` resolution with batch size `16` for `50` epochs starting from pretrained `yolov5s.pt` weights.

In [ ]:
# Run YOLOv5 training
!python train.py --img 416 --batch 16 --epochs 50 --data data.yaml --weights yolov5s.pt --cache

## 5. Evaluate Validation Results
Check precision, recall, mAP@0.5, and confusion matrices generated in `runs/train/exp/`.

In [ ]:
# Display confusion matrix and results curves
Image(filename='runs/train/exp/confusion_matrix.png', width=800)
Image(filename='runs/train/exp/results.png', width=800)

## 6. Test Inference
Test detection on validation images using the newly trained `best.pt` weights.

In [ ]:
!python detect.py --weights runs/train/exp/weights/best.pt --img 416 --conf 0.5 --source ../test/images
import glob
for image_path in glob.glob('runs/detect/exp/*.jpg')[:3]:
    display(Image(filename=image_path, width=500))

## 7. Download `best.pt` for Web App Deployment
Download `best.pt` and place it into the `yolov5/best.pt` directory in your local project.

In [ ]:
from google.colab import files
files.download('runs/train/exp/weights/best.pt')